# Heart Disease Prediction - Data Preprocessing (Day 2)

This notebook covers the data preprocessing pipeline for the combined UCI Heart Disease dataset. We will clean missing values, encode categorical attributes, split the data into training and test sets, and scale features for modeling.

## Step 1: Import Libraries

**What & Why:**
- **What:** Imported libraries for handling arrays/frames (`pandas`, `numpy`), partitioning data (`train_test_split`), and preparing features (`LabelEncoder`, `StandardScaler`).
- **Why:** These form the core modules required to transform raw clinical data into scaled matrices compatible with machine learning algorithms.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

## Step 2: Load the Dataset

**What & Why:**
- **What:** Loaded the cleaned dataset from Day 1 (`heart_clean_day1.csv`) and printed its first 5 rows.
- **Why:** To start preprocessing from a consistent, structured state where identifiers and original multi-class labels have already been removed.

In [ ]:
df = pd.read_csv("../data/heart_clean_day1.csv")
df.head()

## Step 3: Check Missing Values

**What & Why:**
- **What:** Inspected columns to see which ones contain missing values.
- **Why:** To double-check the volume of missing entries before applying imputation strategies.

In [ ]:
df.isnull().sum()

## Step 4: Handle Missing Values

**What, Why & Observations:**
- **What:** Filled missing values in numerical columns using the median and categorical columns using the mode.
- **Why:**
  - *Median Imputation:* Numerical features often contain outliers (e.g. cholesterol values of 0 or high blood pressure). The median is far more robust to these skewing elements than the mean.
  - *Mode Imputation:* Fills missing categorical entries using the most frequently occurring value in the dataset.
- **Observations:** After running imputation, the missing values sum is 0 across all features.

In [ ]:
# Impute numerical columns with median
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Impute categorical columns with mode
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Check for missing values again
df.isnull().sum()

## Step 5: Encode Categorical Features

**What, Why & Observations:**
- **What:** Identified categorical string columns and converted them to numerical labels using `LabelEncoder`.
- **Why:** Machine learning estimators require purely numerical matrix inputs to compute calculations and cost functions.
- **Observations:** Running encoding converts string variables (e.g. Sex: Male/Female) into integer representations (e.g. 1/0).

In [ ]:
# Check categorical columns
print("Categorical columns:", list(df.select_dtypes(include="object").columns))

# Apply LabelEncoder
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# Verify conversion
df.head()

## Step 6: Separate Features and Target

**What & Why:**
- **What:** Partitioned the dataset into feature matrix `X` and label vector `y`.
- **Why:** Separation of predictor attributes from the classification outcome targets is required by scikit-learn models.

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

print(X.shape)
print(y.shape)

## Step 7: Train-Test Split

**What & Why:**
- **What:** Split features and labels into training (80%) and testing (20%) sets using stratification.
- **Why:** Stratified splitting is essential for keeping class distributions balanced in both subsets, preventing representative bias in performance scores.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Step 8: Feature Scaling

**What & Why:**
- **What:** Scaled features to standard normal distribution (mean=0, variance=1) using `StandardScaler`.
- **Why:** Distances and coefficients in ML algorithms (like SVMs, Logistic Regression, and k-NN) are sensitive to different attribute scales, making normalization necessary.
- **Important:** Scaler parameters (`mean`, `std`) are fitted strictly on `X_train` to prevent data leakage from the test set.

In [ ]:
scaler = StandardScaler()

# Fit and transform on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

## Step 9: Verify Shapes

**What, Why & Observations:**
- **What:** Printed shapes of training/testing features and labels.
- **Why:** Final verification that shapes are correct and stratified splitting aligns with expectations.
- **Observations:** Training contains 736 records and testing contains 184 records, matching the expected 80/20 partition.

In [ ]:
print("Training Features:", X_train_scaled.shape)
print("Testing Features :", X_test_scaled.shape)

print("Training Labels :", y_train.shape)
print("Testing Labels  :", y_test.shape)

## Step 10: Save Processed Data

**What & Why:**
- **What:** Saved the final processed DataFrame to `heart_processed.csv`, and saved the scaled features and labels as NumPy binary files (`X_train.npy`, `X_test.npy`, `y_train.npy`, `y_test.npy`).
- **Why:** To persist the preprocessed data on disk for quick, direct loading in modeling tasks, avoiding the need to execute the preprocessing steps again.

In [ ]:
# Save the cleaned and encoded dataset
df.to_csv("../data/heart_processed.csv", index=False)

# Save the scaled features and target arrays as numpy binaries
import numpy as np

np.save("../data/X_train.npy", X_train_scaled)
np.save("../data/X_test.npy", X_test_scaled)

np.save("../data/y_train.npy", y_train)
np.save("../data/y_test.npy", y_test)

## 📝 Mini Exercise Questions & Answers

**1. How many missing values were found initially?**
- A total of **1,759 missing values** were found across the columns in the cleaned dataset.

**2. Which columns contained missing values?**
- The columns containing missing values are:
  - `trestbps` (59)
  - `chol` (30)
  - `fbs` (90)
  - `restecg` (2)
  - `thalach` (55)
  - `exang` (55)
  - `oldpeak` (62)
  - `slope` (309)
  - `ca` (611)
  - `thal` (486)

**3. Why did we use the median instead of the mean?**
- The median is a robust statistic that is not heavily influenced by outliers or skewed distributions. In our dataset, features like cholesterol (`chol`) contain zero values and extreme values that would skew the mean, making the median a safer representation of central tendency.

**4. Why is label encoding required?**
- Machine learning algorithms (such as scikit-learn models) perform numerical computations and require all input features to be numeric. Label encoding converts categorical text columns (e.g. sex, slope, chest pain) into numeric integers so they can be processed by models.

**5. Why do we split before scaling?**
- We split before scaling to prevent **data leakage**. If scaling is performed on the entire dataset first, metrics like the mean and standard deviation of the test set are leaked into the training set, which artificially boosts validation metrics and leads to overly optimistic evaluations.

**6. Why do we fit the scaler only on the training set?**
- To simulate a true production environment where the test set is completely unseen. The scaler learns parameters (mean and standard deviation) solely from the training set, and we then use those learned parameters to transform the test set.

**7. What is the purpose of stratify=y?**
- Stratification guarantees that the proportions of the target classes (diseased vs healthy patients) are identical in both the training and testing sets. This ensures that the model is trained and evaluated on representative subsets.